Initiall inspictation

In [2]:
import pandas as pd 
sales = pd.read_csv ("ecommerce_sales_analytics_5000.csv")
sales.head()

,order_id,order_date,customer_id,product_category,region,quantity,unit_price,discount,payment_method,delivery_days,customer_rating,revenue
0,10001,1/1/2022,1102,Beauty,South,7,373.65,0.28,Wallet,10,4.7,1883.20
1,10002,1/2/2022,1435,Clothing,South,7,47.74,0.09,Card,6,3.9,304.10
2,10003,1/3/2022,1860,Beauty,East,3,311.28,0.31,COD,6,2.5,644.35
3,10004,1/4/2022,1270,Electronics,West,5,524.47,0.02,Wallet,6,1.6,2569.90
4,10005,1/5/2022,1106,Clothing,West,5,139.87,0.33,Wallet,4,4.9,468.56


QUALITY SUMMARY


inspecting cleaning and validating dataset


In [34]:
#Rows before cleaning 
rows_before = len(sales)
print (f"Rows before cleaning :{rows_before}")
#Basic shape and structure
print(sales.shape)         
print(sales.dtypes)         
print(sales.head())         
print(sales.tail())        

# Missing values 
print(sales.isnull().sum()) 
#checking for  duplicates
n_dupes_full = sales.duplicated().sum()
n_dupes_order_id = sales["order_id"].duplicated().sum()
print(f"Fully duplicated rows: {n_dupes_full}")
print(f"Duplicated order_id values: {n_dupes_order_id}")
sales.loc[sales.duplicated(subset='order_id', keep=False)].sort_values("order_id").head(6)
# Numeric summary statistics
print(sales.describe())
#Categorical breakdowns
for col in ['product_category', 'region', 'payment_method']:
 print(sales[col].value_counts())
#Cuostomer behavior
print(f"Unique customers: {sales['customer_id'].nunique()}")
print(f"Repeat customers: {(sales['customer_id'].value_counts() > 1).sum()}")
#Verifying Revenue Calculation
expected_revenue = (sales["quantity"] * sales["unit_price"] * (1 - sales["discount"])).round(2)
revenue_mismatches = (abs(sales["revenue"] - expected_revenue) > 0.01).sum()
print(f"Revenue mismatches: {revenue_mismatches}")
# validating numeric fields
assert (sales["quantity"] > 0).all()
assert (sales["unit_price"] > 0).all()
assert (sales["discount"].between(0, 1)).all()
assert (sales["customer_rating"].between(1, 5)).all()
assert (sales["delivery_days"] >= 0).all()

Rows before cleaning :5000
(5000, 14)
order_id                         str
order_date            datetime64[us]
customer_id                      str
product_category                 str
region                           str
quantity                       int64
unit_price                   float64
discount                     float64
payment_method                   str
delivery_days                  int64
customer_rating              float64
revenue                      float64
order_date_parsed     datetime64[us]
calculated_revenue           float64
dtype: object
  order_id order_date customer_id product_category region  quantity  \
0    10001 2022-01-01        1102           Beauty  South         7   
1    10002 2022-01-02        1435         Clothing  South         7   
2    10003 2022-01-03        1860           Beauty   East         3   
3    10004 2022-01-04        1270      Electronics   West         5   
4    10005 2022-01-05        1106         Clothing   West         5   

   

standarizing column names

In [35]:
sales.columns = (sales.columns.str.strip().str.lower().str.replace(" ", "_") .str.replace("-", "_")
)

Cleaning and Standardization of Categorical Text Data

In [36]:
#Identify Text Columns
text_cols = sales.select_dtypes(include=["str"]).columns
#Remove Leading and Trailing Whitespace
for col in text_cols:
    sales[col] = sales[col].str.strip()
#Standardize Text Case
for col in text_cols:
    sales[col] = sales[col].str.title()

Data Type Conversion and Standardization

In [37]:

#Convert Date Column to Datetime
sales["order_date"] = pd.to_datetime(sales["order_date"], format="%m/%d/%Y",  errors="coerce")
#Convert ID Columns to String
sales["order_id"] = sales["order_id"].astype(str)
sales["customer_id"] = sales ["customer_id"].astype(str)
#Convert Whole-Number Floats to Integers
for col in sales.select_dtypes(include=["float64"]).columns:
    if sales[col].dropna().apply(lambda x: x == int(x)).all():
        sales[col] = sales[col].astype("Int64")

checking for missing values

In [38]:
missing = sales.isnull().sum()
print ("missing values per column:")
print(missing [missing>0])

missing values per column:
Series([], dtype: int64)


Outlier Detection and Removal Using IQR

In [39]:
# Remove outliers using the IQR method

def remove_outliers_iqr(sales, column, factor=1.5):
    Q1 = sales[column].quantile(0.25)
    Q3 = sales[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    return sales[
        (sales[column] >= lower) &
        (sales[column] <= upper)]
columns_to_clean = [
    "quantity","unit_price","discount","customer_rating","delivery_days"]
for col in columns_to_clean:
    sales = remove_outliers_iqr(sales, col)

Rows after cleaning


In [40]:
print (f"Rows after cleaning :{len(sales)}")

Rows after cleaning :5000


In [43]:

print("DATA QUALITY SUMMARY")
print(f"Rows before cleaning: {rows_before}")
print(f"Rows after cleaning: {len(sales)}")
print(f"Missing revenue: {sales['revenue'].isna().sum()}")
print(f"Missing product: {sales['product_category'].isna().sum()}")
print(f"Duplicate orders: {sales['order_id'].duplicated().sum()}")
print(f"Invalid dates: {sales['order_date'].isna().sum()}")
print(f"Invalid quantities: {(sales['quantity'] <= 0).sum()}")
print(f"Invalid unit prices: {(sales['unit_price'] <= 0).sum()}")
print(f"Unique regions: {sales['region'].nunique()}")
print(sales["region"].unique())
print(f"Revenue mismatches: {revenue_mismatches}")

DATA QUALITY SUMMARY
Rows before cleaning: 5000
Rows after cleaning: 5000
Missing revenue: 0
Missing product: 0
Duplicate orders: 0
Invalid dates: 0
Invalid quantities: 0
Invalid unit prices: 0
Unique regions: 4
<StringArray>
['South', 'East', 'West', 'North']
Length: 4, dtype: str
Revenue mismatches: 0


In [44]:
sales. to_csv("sales_cleaned.csv",index=False)